# HW2：King County 房價多元線性回歸（主程式）

本 notebook 是 HW2 的主程式，也是報告、README 與 dashboard 全部數字的唯一來源。它依 CRISP-DM 六個階段組織，呼叫單元內的 `hw2` 套件，並把結果匯出到 `results/` 與 `assets/model_data.js`。

- 在 repo 內的檔名是 `hw2.ipynb`；繳交時由打包步驟從同一份檔案複製成 `<學號>_hw2.ipynb`。
- 執行方式見單元 `README.md`（從 `home_work_02/` 以非互動方式執行全部 cell）。
- 固定 seed、鎖版本的 `requirements.txt`：在乾淨環境重跑可得到相同的切分與數字。

In [1]:
import sys
from pathlib import Path

UNIT_DIR = Path.cwd()
if not (UNIT_DIR / "hw2").is_dir():
    raise RuntimeError("run this notebook from home_work_02/")
sys.path.insert(0, str(UNIT_DIR))

import numpy as np
import pandas as pd

from hw2 import contract, data, export, paths, preprocess

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
export.package_versions()

{'python': '3.11.2',
 'numpy': '2.4.6',
 'pandas': '3.0.6',
 'scipy': '1.17.1',
 'scikit-learn': '1.9.1',
 'matplotlib': '3.11.2'}

## 1. Business Understanding

**目標**：以 King County（美國華盛頓州，含 Seattle）2014-05 至 2015-05 的房屋成交紀錄，建立多元線性回歸模型預測成交價，並為每一筆預測提供有統計定義的 95% 預測區間。

**研究問題**

1. 在四種具有 k = 1…18 路徑的特徵選擇方法（Forward SFS、Backward SFS、RFE、Lasso 路徑排序取 top-k）與單一的 All-in baseline 之中，哪一個在訓練集交叉驗證下的預測誤差最小？
2. 被選出的模型在保留的 Test set 上表現如何？逐筆 95% 預測區間的經驗涵蓋率是多少？
3. 哪些房屋特徵與價格的關聯最強？模型的假設與限制是什麼？

**成功標準**

- 模型選擇只用訓練集：以房屋 `id` 分組的 5-fold 交叉驗證、USD 尺度的 CV RMSE。
- Test set 在全部選擇固定之後只評估一次，報告 MAE、MSE、RMSE、R²（USD）與預測區間的涵蓋率；不以 Test 結果更換模型。
- 結果可重現：固定 seed、鎖版本的套件、資料檔以 sha256 核對。

**資料來源**：Kaggle「House Sales in King County, USA」（`harlfoxem/housesalesprediction`），授權 CC0: Public Domain。本專案使用的檔案是 acceptor 於 2026-10-18 提供的 GitHub 鏡像，其 sha256 與 Brief 釘住的值相同（下方程式核對）。

**特徵數的規則**：作業頁正文寫「至 Kaggle 選擇一個具有 10 至 20 個特徵 (features) 的公開資料集。」，補充段寫「choose features > 13, < 40」。本專案採兩者的交集 14–20，在類別編碼前計數、不含識別欄 `id` 與目標 `price`：

- **Source features 19 個**（CSV 21 欄扣除 `id`、`price`），落在 14–20 之內。
- **建模 feature groups 18 個**：`sqft_basement` 不進模型，因為每一列都有 `sqft_living = sqft_above + sqft_basement`，三欄完全共線。特徵選擇以 feature group 為單位（例如 `zipcode` 與它的全部 one-hot 欄同進同出），k 計的是 group 數。

In [2]:
print(data.DATASET["name"], "|", data.DATASET["kaggle_slug"], "|", data.DATASET["license"])
print("source features:", len(preprocess.SOURCE_FEATURES), list(preprocess.SOURCE_FEATURES))
print("feature groups :", len(preprocess.FEATURE_GROUPS), list(preprocess.FEATURE_GROUPS))
print("excluded       :", preprocess.EXCLUDED_SOURCE_FEATURES)

House Sales in King County, USA | harlfoxem/housesalesprediction | CC0: Public Domain (CC0-1.0)
source features: 19 ['date', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long', 'sqft_living15', 'sqft_lot15']
feature groups : 18 ['date', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long', 'sqft_living15', 'sqft_lot15']
excluded       : {'sqft_basement': 'sqft_living = sqft_above + sqft_basement on every row, so the three columns are perfectly collinear; sqft_basement is dropped and the other two stay'}


## 2. Data Understanding

### 2.1 全資料層級：只做結構驗證

先以 sha256 核對資料檔，再解析 CSV，並檢查 schema（21 欄與欄位順序）、列數 21,613、唯一 `id` 數 21,436、沒有缺值，以及每一列的 `sqft_living = sqft_above + sqft_basement`。任一項不符就以錯誤停止。

全資料層級只做這些結構檢查。接著立即以房屋 `id` 分組切出 Test set：資料階段只回傳 training 列與 Test 列的識別（CSV 列序號與 `id`）及 n_test，不回傳 Test 列的特徵或價格。價格與特徵的分布統計、極端值都只用 training 列計算。

In [3]:
prepared = data.load_and_split()
print("sha256        :", prepared.sha256)
print("structure     :", prepared.facts)
print("n_train/n_test:", prepared.n_train, prepared.n_test,
      f"(test share {prepared.n_test / (prepared.n_train + prepared.n_test):.4f})")
print("timing (s)    :", prepared.timings_s)

sha256        : d0875baa0251b21d4bdc9d2ae940a4fe0bb6009824f23dd0e2a5b2bf04557b7e
structure     : {'n_rows': 21613, 'n_columns': 21, 'n_unique_ids': 21436, 'n_repeat_sales': 177, 'sqft_identity_holds': True}
n_train/n_test: 17290 4323 (test share 0.2000)
timing (s)    : {'data_load_validate': 0.0367, 'split_and_folds': 0.0167}


### 2.2 Training set 的分布（只用 training 列）

以下統計全部來自 training 列。價格分布右偏，所以模型以 ln(price) 為目標，預測與指標再轉回 USD 報告。

In [4]:
train = prepared.training
price = pd.DataFrame({"price (USD)": train["price"], "ln(price)": np.log(train["price"])})
price.describe().T

,count,mean,std,min,25%,50%,75%,max
price (USD),17290.0,537754.460150,360059.981568,75000.000000,320000.000000,450000.000000,641000.000000,7.700000e+06
ln(price),17290.0,13.045136,0.525038,11.225243,12.676076,13.017003,13.370785,1.585673e+01


In [5]:
numeric = [f for f in preprocess.SOURCE_FEATURES if f not in ("date", "zipcode")]
train[numeric].describe().T

,count,mean,std,min,25%,50%,75%,max
bedrooms,17290.0,3.371718,0.935417,0.0000,3.0000,3.0000,4.0000,3.300000e+01
bathrooms,17290.0,2.109905,0.769467,0.0000,1.5000,2.2500,2.5000,8.000000e+00
sqft_living,17290.0,2075.460844,914.613101,290.0000,1420.0000,1910.0000,2550.0000,1.354000e+04
sqft_lot,17290.0,15065.460266,39361.979524,520.0000,5056.5000,7620.0000,10712.5000,1.164794e+06
floors,17290.0,1.494043,0.540022,1.0000,1.0000,1.5000,2.0000,3.500000e+00
waterfront,17290.0,0.007403,0.085725,0.0000,0.0000,0.0000,0.0000,1.000000e+00
view,17290.0,0.232042,0.761708,0.0000,0.0000,0.0000,0.0000,4.000000e+00
condition,17290.0,3.411625,0.649326,1.0000,3.0000,3.0000,4.0000,5.000000e+00
grade,17290.0,7.650839,1.171092,1.0000,7.0000,7.0000,8.0000,1.300000e+01
sqft_above,17290.0,1785.273569,824.804018,290.0000,1190.0000,1560.0000,2210.0000,9.410000e+03


In [6]:
days = preprocess.sale_days(train["date"])
print("sale dates     :", preprocess.days_to_iso(days.min()), "to", preprocess.days_to_iso(days.max()))
print("zipcodes       :", train["zipcode"].nunique(), "categories in the training rows")
print("renovated rows :", int((train["yr_renovated"] > 0).sum()), "of", len(train))
n_houses = train["id"].nunique()
print("training houses:", n_houses, "| repeat sales in training:", len(train) - n_houses)

sale dates     : 2014-05-02 to 2015-05-27
zipcodes       : 70 categories in the training rows
renovated rows : 716 of 17290
training houses: 17149 | repeat sales in training: 141


### 2.3 極端值（只用 training 列；不刪除）

異常值不刪除也不截尾；這裡列出 training 列中的極端值，之後在模型診斷中討論它們的影響。

In [7]:
cols = ["row", "id", "date", "price", "bedrooms", "bathrooms", "sqft_living", "sqft_lot", "grade", "zipcode"]
print("highest prices"); display(train.nlargest(5, "price")[cols])
print("lowest prices"); display(train.nsmallest(5, "price")[cols])
print("most bedrooms"); display(train.nlargest(3, "bedrooms")[cols])
print("largest living area"); display(train.nlargest(3, "sqft_living")[cols])

highest prices


,row,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,grade,zipcode
5784,7252,6762700020,20141013T000000,7700000.0,6,8.0,12050,27600,13,98102
3145,3914,9808700762,20140611T000000,7062500.0,5,4.5,10040,37325,11,98004
1053,1315,7558700030,20150413T000000,5300000.0,6,6.0,7390,24829,12,98040
2120,2626,7738500731,20140815T000000,4500000.0,5,5.5,6640,40014,12,98155
6869,8638,3835500195,20140618T000000,4489000.0,4,3.0,6430,27517,12,98004


lowest prices


,row,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,grade,zipcode
913,1149,3421079032,20150217T000000,75000.0,1,0.0,670,43377,3,98022
12263,15293,0040000362,20140506T000000,78000.0,2,1.0,780,16344,5,98168
12981,16198,3028200080,20150324T000000,81000.0,2,1.0,730,9975,5,98168
6585,8274,3883800011,20141105T000000,82000.0,3,1.0,860,10426,6,98146
1735,2141,1623049041,20140508T000000,82500.0,2,1.0,520,22334,5,98168


most bedrooms


,row,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,grade,zipcode
12730,15870,2402100895,20140625T000000,640000.0,33,1.75,1620,6000,7,98103
6966,8757,1773100755,20140821T000000,520000.0,11,3.00,3000,4960,7,98106
10672,13314,0627300145,20140814T000000,1148000.0,10,5.25,4590,10920,9,98004


largest living area


,row,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,grade,zipcode
10238,12777,1225069038,20140505T000000,2280000.0,7,8.0,13540,307752,12,98053
5784,7252,6762700020,20141013T000000,7700000.0,6,8.0,12050,27600,13,98102
3145,3914,9808700762,20140611T000000,7062500.0,5,4.5,10040,37325,11,98004


## 3. Data Preparation

### 3.1 固定的逐列轉換（不需 fit，對所有候選相同）

- `date` → `sale_days`：距固定基準日 2014-01-01 的天數。
- `yr_renovated` → `is_renovated`（有整修為 1）與 `yr_renovated_effective`（有整修時為整修年，否則為 `yr_built`）；兩欄同屬 `yr_renovated` 這個 group。
- `zipcode` 視為類別（nominal）。
- `sqft_basement` 不進模型；其餘欄位照原值。

In [8]:
example = train.head(3)[["row", "date", "yr_built", "yr_renovated", "zipcode"]].copy()
example["sale_days"] = preprocess.sale_days(example["date"])
example["is_renovated"], example["yr_renovated_effective"] = preprocess.renovation_columns(
    example["yr_renovated"], example["yr_built"])
example

,row,date,yr_built,yr_renovated,zipcode,sale_days,is_renovated,yr_renovated_effective
0,0,20141013T000000,1955,0,98178,285,0,1955
1,1,20141209T000000,1951,1991,98125,342,1,1991
2,2,20150225T000000,1933,0,98028,420,0,1933


### 3.2 需要 fit 的前處理（只以當次訓練列 fit）

- `zipcode` one-hot：類別取自當次訓練列，排序後的第一個類別是基準類別、不建欄；預測時遇到未見類別，編為基準類別。
- 數值欄的標準化只用於 Ridge／Lasso；OLS 子集模型以原始尺度擬合，係數可以直接解讀。

在交叉驗證中，每個 fold 都只用該 fold 的訓練列 fit；下面是以整個 training set fit 的結果，供 dashboard 與匯出使用。

In [9]:
full_fit = preprocess.fit_preprocessor(train)
print("encoded columns:", len(full_fit.columns))
print("zipcode categories:", len(full_fit.zipcode_categories), "| baseline:", full_fit.zipcode_baseline)
print(full_fit.columns[:20], "...")

encoded columns: 87
zipcode categories: 70 | baseline: 98001
('sale_days', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'yr_built', 'is_renovated', 'yr_renovated_effective', 'zipcode_98002', 'zipcode_98003', 'zipcode_98004', 'zipcode_98005', 'zipcode_98006', 'zipcode_98007') ...


### 3.3 分組交叉驗證的 fold

在 training set 內以房屋 `id` 分組產生 5 個 fold，只產生一次，所有方法、所有 k 與 Ridge／Lasso 都共用同一組 fold。

In [10]:
folds = train.groupby("fold").agg(rows=("row", "size"), houses=("id", "nunique"))
folds["price median (USD)"] = train.groupby("fold")["price"].median()
houses = [set(train.loc[train["fold"] == f, "id"]) for f in range(prepared.n_folds)]
assert all(houses[a].isdisjoint(houses[b]) for a in range(5) for b in range(a + 1, 5))
folds

,rows,houses,price median (USD)
fold,,,
0,3454,3430,453000.0
1,3462,3430,450000.0
2,3461,3430,449400.0
3,3456,3430,451200.5
4,3457,3429,449999.0


### 3.4 匯出 `model_data`

這一步寫出 `model_data` 的 `meta`、`preprocess`、`inputs`，以及 `test_inputs` 的結構（沒有任何列）。Test 列的特徵與價格只會在全部模型選擇固定之後、由一次性的 Test 評估一併寫出。`results/split/split_assignment.csv` 記錄每一筆交易屬於 training 或 Test 與其 fold 編號（只有識別，沒有特徵或價格）。

In [11]:
model_data = export.build_model_data(prepared, fixture=False)
json_path, js_path = export.write_model_data(model_data)
split_path = export.write_split_assignment(prepared)
for p in (json_path, js_path, split_path):
    print(p.relative_to(UNIT_DIR).as_posix(), p.stat().st_size, "bytes")
print("sections:", list(model_data))
print("timing (s):", model_data["meta"]["timing_s"])

results/model_data.json 15859 bytes
assets/model_data.js 15945 bytes
results/split/split_assignment.csv 520585 bytes
sections: ['meta', 'preprocess', 'inputs', 'test_inputs']
timing (s): {'data_load_validate': 0.0367, 'split_and_folds': 0.0167}


## 4. Modeling

（本章尚未完成：特徵選擇路徑、All-in baseline、Ridge／Lasso 對照與講義的 p-value backward 對照會在後續步驟加入。）

## 5. Evaluation

（本章尚未完成：CV winner、一次性的 Test 評估與預測區間的涵蓋率會在後續步驟加入。）

## 6. Deployment

（本章尚未完成：最終模型、`predict_price()` 示範、模型解釋與使用限制會在後續步驟加入。）